In [1]:
import requests
import time
import pandas as pd
import os
import json
import ast

In [2]:
# For PGS training inforamtion

def get_method(score: json) -> str:
    temp = score.get("method_name", "None")
    return str(temp)

def get_variants_number(score: json) -> int:
    temp = score.get("variants_number", 0)
    return temp

def get_date_release(score: json) -> str:
    temp = score.get("date_release", "None")
    return str(temp)

def get_sample_number(score: json) -> int:

    if len(score.get("samples_variants", [])) > 0:
        temp = score.get("samples_variants", [])
        sample_num = temp[0].get("sample_number", 0)
        if not sample_num == "null":
            return sample_num
    
    elif len(score.get("samples_training", [])) > 0:
        temp = score.get("samples_training", [])
        sample_num = temp[0].get("sample_number", 0)
        if not sample_num == "null":
            return sample_num
    
    return 0

def get_sample_case(score: json) -> int:

    if len(score.get("samples_variants", [])) > 0:
        temp = score.get("samples_variants", [])
        sample_case = temp[0].get("sample_cases", 0)
        if not sample_case == "null":
            return sample_case
    
    elif len(score.get("samples_training", [])) > 0:
        temp = score.get("samples_training", [])
        sample_case = temp[0].get("sample_cases", 0)
        if not sample_case == "null":
            return sample_case
    
    return 0

def get_sample_controls(score: json) -> int:

    if len(score.get("samples_variants", [])) > 0:
        temp = score.get("samples_variants", [])
        sample_controls = temp[0].get("sample_controls", 0)
        if not sample_controls == "null":
            return sample_controls
    
    elif len(score.get("samples_training", [])) > 0:
        temp = score.get("samples_training", [])
        sample_controls = temp[0].get("sample_controls", 0)
        if not sample_controls == "null":
            return sample_controls
    
    return 0

def get_ancestry_broad(score: json) -> str:

    if len(score.get("samples_variants", [])) > 0:
        temp = score.get("samples_variants", [])
        sample_ancestry = temp[0].get("ancestry_broad", "None")
        if not sample_ancestry == "Not reported":
            return sample_ancestry
    
    elif len(score.get("samples_training", [])) > 0:
        temp = score.get("samples_training", [])
        sample_ancestry = temp[0].get("ancestry_broad", "None")
        if not sample_ancestry == "Not reported":
            return sample_ancestry
    
    return "None"

def get_cohort(score: json) -> str:

    if len(score.get("samples_variants", [])) > 0:
        temp = score.get("samples_variants", [])
        if len(temp[0].get("cohorts", [])) > 0:
            temp = temp[0].get("cohorts", [])
            return temp[0].get("name_full", "None")

    elif len(score.get("samples_training", [])) > 0:
        temp = score.get("samples_training", [])
        if len(temp[0].get("cohorts", [])) > 0:
            temp = temp[0].get("cohorts", [])
            return temp[0].get("name_full", "None")
    
    return "None"

In [3]:
# For PGS evaluation inforamtion 
def get_eval_num(score: json) -> int:
      temp = score.get("count", 0)
      return int(temp)

def get_eval_sample(score: json) -> list:
      result = []
      temp = score.get("results", [])
      if len(temp) == 0:
            return result
      for t in temp:
            temp2 = t.get("sampleset").get("samples")[0]
            result.append(temp2.get("sample_number"))
      return result

def get_eval_control(score: json) -> list:
      result = []
      temp = score.get("results", [])
      if len(temp) == 0:
            return result
      for t in temp:
            temp2 = t.get("sampleset").get("samples")[0]
            result.append(temp2.get("sample_controls"))
      return result

def get_eval_case(score: json) -> list:
      result = []
      temp = score.get("results", [])
      if len(temp) == 0:
            return result
      for t in temp:
            temp2 = t.get("sampleset").get("samples")[0]
            result.append(temp2.get("sample_cases"))
      return result

def get_eval_ancestry(score: json) -> list:
      result = []
      temp = score.get("results", [])
      if len(temp) == 0:
            return result
      for t in temp:
            temp2 = t.get("sampleset").get("samples")[0]
            result.append(temp2.get("ancestry_broad"))
      return result

def get_eval_cohort(score: json) -> list:
      result = []
      temp = score.get("results", [])
      if len(temp) == 0:
            return result
      for t in temp:
            temp2 = t.get("sampleset").get("samples")[0]
            temp3 = temp2.get("cohorts", [])
            if len(temp3) != 0:
                  result.append(temp3[0].get("name_full"))
                  continue
            result.append("None")
      return result

def get_eval_metrics(score: json) -> list:
      result = []
      temp = score.get("results", [])
      if len(temp) == 0:
            return result
      for t in temp:
            temp2 = t.get("performance_metrics").get("effect_sizes", [])
            if len(temp2) != 0:
                  result.append(temp2[0].get("name_long"))
                  continue
            temp2 = t.get("performance_metrics").get("class_acc", [])
            if len(temp2) != 0:
                  result.append(temp2[0].get("name_long"))
                  continue
            temp2 = t.get("performance_metrics").get("othermetrics", [])
            if len(temp2) != 0:
                  result.append(temp2[0].get("name_long"))
                  continue
            result.append("None")
      return result

def get_eval_covariates(score: json) -> list:
      result = []
      temp = score.get("results", [])
      if len(temp) == 0:
            return result
      for t in temp:
            result.append(t.get("covariates"))
      return result

def get_eval_estimate(score: json) -> list:
      result = []
      temp = score.get("results", [])
      if len(temp) == 0:
            return result
      for t in temp:
            temp2 = t.get("performance_metrics").get("effect_sizes", [])
            if len(temp2) != 0:
                  result.append(temp2[0].get("estimate", 0))
                  continue
            temp2 = t.get("performance_metrics").get("class_acc", [])
            if len(temp2) != 0:
                  result.append(temp2[0].get("estimate", 0))
                  continue
            temp2 = t.get("performance_metrics").get("othermetrics", [])
            if len(temp2) != 0:
                  result.append(temp2[0].get("estimate", 0))
                  continue
            result.append(0)
      return result


In [4]:
# Read contained icd and description
pgs_id_list = pd.read_csv(os.path.join(os.getcwd(), "pgs_id_list_binary_combined.csv"), index_col=None)

records = []
processed_pgs = []
for idx, row in pgs_id_list.iterrows():
    pgs_ids_str = row["pgs_ids"]
    pgs_ids = ast.literal_eval(pgs_ids_str)

    for pgs_id in pgs_ids:
        if pgs_id in processed_pgs:
            continue
        temp = "https://www.pgscatalog.org/rest/score/" + pgs_id
        r = requests.get(temp)
        data = r.json()

        temp = "https://www.pgscatalog.org/rest/performance/search?pgs_id=" + pgs_id
        r = requests.get(temp)
        perform_data = r.json()

        records.append({
            "PGS_ID": pgs_id,
            "num_variant": get_variants_number(data),
            "training_ancestry": get_ancestry_broad(data),
            "training_method": get_method(data),
            "training_cohort": get_cohort(data),
            "num_training_sample": get_sample_number(data),
            "num_training_controls": get_sample_controls(data),
            "num_training_cases": get_sample_case(data),
            "date_release": get_date_release(data),
            "num_eval": get_eval_num(perform_data),
            "num_eval_sample": get_eval_sample(perform_data),
            "num_eval_controls": get_eval_control(perform_data),
            "num_eval_cases": get_eval_case(perform_data),
            "eval_ancestry": get_eval_ancestry(perform_data),
            "eval_cohort": get_eval_cohort(perform_data),
            "eval_metrics": get_eval_metrics(perform_data),
            "eval_covariates": get_eval_covariates(perform_data),
            "eval_estimate": get_eval_estimate(perform_data)
        })
        processed_pgs.append(pgs_id)
        time.sleep(0.2)
    print(f"Finish processing {row['icd']}")

df = pd.DataFrame(records)
output_path = "pgs_metadata_binary_combined.csv"
df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Finish processing I714
Finish processing R9431
Finish processing M0579
Finish processing M0579
Finish processing N179
Finish processing C9100
Finish processing I219
Finish processing H905
Finish processing H353131
Finish processing F1020
Finish processing F1099
Finish processing J309
Finish processing G309
Finish processing D649
Finish processing I208
Finish processing M459
Finish processing F419
Finish processing I350
Finish processing I069
Finish processing M1990
Finish processing J45909
Finish processing L2084
Finish processing I4891
Finish processing I4892
Finish processing F909
Finish processing F840
Finish processing C4491
Finish processing N400
Finish processing E806
Finish processing F319
Finish processing F3181
Finish processing D689
Finish processing M844
Finish processing C509
Finish processing C3490
Finish processing I498
Finish processing I429
Finish processing H2513
Finish processing L0390
Finish processing C539
Finish processing N189
Finish processing J449
Finish process

In [9]:
# Re-fetch metadata for rows where num_eval == 0
df = pd.read_csv("pgs_metadata_binary_combined.csv", encoding='latin-1')
zero_variant_ids = df[df["num_training_sample"].isna()]["PGS_ID"].tolist()
print(f"Found {len(zero_variant_ids)} PGS with num_eval == 0: {zero_variant_ids}")

for pgs_id in zero_variant_ids:
    print(f"Re-fetching {pgs_id}...")
    try:
        r = requests.get(f"https://www.pgscatalog.org/rest/score/{pgs_id}")
        data = r.json()
        r2 = requests.get(f"https://www.pgscatalog.org/rest/performance/search?pgs_id={pgs_id}")
        perform_data = r2.json()

        new_vals = {
            "num_variant": get_variants_number(data),
            "training_ancestry": get_ancestry_broad(data),
            "training_method": get_method(data),
            "training_cohort": get_cohort(data),
            "num_training_sample": get_sample_number(data),
            "num_training_controls": get_sample_controls(data),
            "num_training_cases": get_sample_case(data),
            "date_release": get_date_release(data),
            "num_eval": get_eval_num(perform_data),
            "num_eval_sample": get_eval_sample(perform_data),
            "num_eval_controls": get_eval_control(perform_data),
            "num_eval_cases": get_eval_case(perform_data),
            "eval_ancestry": get_eval_ancestry(perform_data),
            "eval_cohort": get_eval_cohort(perform_data),
            "eval_metrics": get_eval_metrics(perform_data),
            "eval_covariates": get_eval_covariates(perform_data),
            "eval_estimate": get_eval_estimate(perform_data),
        }

        new_vals_str = {col: str(val) if isinstance(val, list) else val for col, val in new_vals.items()}
        df.loc[df["PGS_ID"] == pgs_id, list(new_vals_str.keys())] = list(new_vals_str.values())
        
        time.sleep(0.3)

    except Exception as e:
        print(f"  {pgs_id}: failed ({e})")

df.to_csv("pgs_metadata_binary_combined.csv", index=False)
print("Saved updated CSV.")

Found 24 PGS with num_eval == 0: ['PGS002731', 'PGS000343', 'PGS002735', 'PGS002737', 'PGS002736', 'PGS000764', 'PGS005265', 'PGS005266', 'PGS005264', 'PGS005272', 'PGS005271', 'PGS005270', 'PGS005269', 'PGS005268', 'PGS005267', 'PGS002291', 'PGS005274', 'PGS005263', 'PGS005273', 'PGS005261', 'PGS005262', 'PGS005260', 'PGS005259', 'PGS005258']
Re-fetching PGS002731...
Re-fetching PGS000343...
Re-fetching PGS002735...
Re-fetching PGS002737...
Re-fetching PGS002736...
Re-fetching PGS000764...
Re-fetching PGS005265...
Re-fetching PGS005266...
Re-fetching PGS005264...
Re-fetching PGS005272...
Re-fetching PGS005271...
Re-fetching PGS005270...
Re-fetching PGS005269...
Re-fetching PGS005268...
Re-fetching PGS005267...
Re-fetching PGS002291...
Re-fetching PGS005274...
Re-fetching PGS005263...
Re-fetching PGS005273...
Re-fetching PGS005261...
Re-fetching PGS005262...
Re-fetching PGS005260...
Re-fetching PGS005259...
Re-fetching PGS005258...
Saved updated CSV.


In [6]:
temp = "https://www.pgscatalog.org/rest/performance/search?pgs_id=PGS000427"
r = requests.get(temp)
score = r.json()
print(json.dumps(score, indent=2, ensure_ascii=False))

{
  "size": 1,
  "count": 1,
  "next": null,
  "previous": null,
  "results": [
    {
      "id": "PPM001112",
      "associated_pgs_id": "PGS000427",
      "phenotyping_reported": "Melanomas of skin, dx or hx",
      "publication": {
        "id": "PGP000118",
        "title": "Cancer PRSweb: An Online Repository with Polygenic Risk Scores for Major Cancer Traits and Their Evaluation in Two Independent Biobanks.",
        "doi": "10.1016/j.ajhg.2020.08.025",
        "PMID": 32991828,
        "journal": "Am J Hum Genet",
        "firstauthor": "Fritsche LG",
        "date_publication": "2020-09-28"
      },
      "sampleset": {
        "id": "PSS000543",
        "samples": [
          {
            "sample_number": 11974,
            "sample_cases": 1325,
            "sample_controls": 10649,
            "sample_percent_male": null,
            "sample_age": null,
            "phenotyping_free": "PheCode:172.1; ICD9CM:172.0, 172.1, 172.2, 172.3, 172.4, 172.5, 172.6, 172.7, 172.8, 172.9

In [7]:
temp = "https://www.pgscatalog.org/rest/score/PGS005265"
r = requests.get(temp)
data = r.json()
print(json.dumps(data, indent=2, ensure_ascii=False))

{
  "id": "PGS005265",
  "name": "graves_disease_mixed_prscs",
  "ftp_scoring_file": "https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS005265/ScoringFiles/PGS005265.txt.gz",
  "ftp_harmonized_scoring_files": {
    "GRCh37": {
      "positions": "https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS005265/ScoringFiles/Harmonized/PGS005265_hmPOS_GRCh37.txt.gz"
    },
    "GRCh38": {
      "positions": "https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS005265/ScoringFiles/Harmonized/PGS005265_hmPOS_GRCh38.txt.gz"
    }
  },
  "publication": {
    "id": "PGP000748",
    "title": "Global multi-ancestry genetic study elucidates genes and biological pathways associated with thyroid cancer and benign thyroid diseases",
    "doi": "10.1101/2025.05.15.25327513",
    "PMID": null,
    "journal": "medRxiv",
    "firstauthor": "White SL",
    "date_publication": "2025-05-16"
  },
  "matches_publication": true,
  "samples_variants": [
    {
      "sample_number": null,
      "sample_cas

In [8]:
print(get_sample_number(data))

None
